# ChipWhisperer 입문 강의자료 — 2강

## 오류주입 공격(Fault Injection Attack) 플랫폼 실습 — 입문

---

### 🎯 강의 목표

이 노트북은 **ChipWhisperer로 오류주입 공격(FIA, Fault Injection Attack)을 처음 접하는 사용자**를 대상으로 합니다.
글리치 파라미터를 이해하고, 광범위한 탐색을 통해 **최적의 글리치 파라미터를 찾는 입문 절차**까지를 단계별로 학습합니다.

| 단계 | 내용 | 핵심 산출물 |
|:----:|:----|:----|
| **1단계** | 글리치 모듈 설정 + ADC 클럭 동기화 (1 sample = 1 clock) | `cglitch_setup()` 호출, 클럭 정렬 |
| **2단계** | 베이스라인 파형 수집 및 글리치 시점 후보 식별 | `expected_ret`, `trace`, `phase_shift_steps` |
| **3단계** | 광범위 파라미터 탐색 + 결과 6단계 분류 | `cglitch_result` 배열 |
| **4단계** | 통계 분석으로 최적 파라미터 도출 | `(i_offset, i_width)` 모드값 |
| **5단계** | Bokeh으로 파라미터 분포 시각화 | 인터랙티브 산점도 |

> 본 강의는 **클럭 글리치(Clock Glitch)** 를 기준으로 진행합니다.

---

## 🔍 시작하기 전에

### 오류주입 공격(Fault Injection Attack)이란?

암호 알고리즘이 수학적으로는 안전하더라도, 동작 중인 하드웨어에 **물리적 외란(perturbation)** 을 가하면
연산 결과가 잘못 출력되거나 보안 기능이 우회될 수 있습니다.
이 누설을 의도적으로 유도해 비밀 정보를 추출하거나 보안 검증을 무력화하는 기법을 **오류주입 공격** 이라 부릅니다.

#### SCA vs FIA — 한눈에 비교

| 항목 | 부채널 분석(SCA) | 오류주입 공격(FIA) |
|:----:|:----:|:----:|
| 공격 형태 | **수동 (passive)** — 관찰만 함 | **능동 (active)** — 외란을 가함 |
| 측정 대상 | 전력/EM/타이밍 | 출력 결과의 오류 패턴 |
| 분석 방식 | 통계적 상관 분석 | 차분 결함 분석(DFA), 명령어 스킵 등 |
| 위험성 | 비파괴적 | 타겟이 망가질 수 있음 ⚠️ |

#### 대표적인 FIA 기법

```
┌─────────────────────────────────────────────────────────────┐
│   1) 클럭 글리치 (Clock Glitch)   ← 본 강의의 주제         │
│      → 클럭 신호에 짧은 추가 펄스 삽입 → 셋업 타임 위반     │
│                                                             │
│   2) 전압 글리치 (Voltage Glitch)                           │
│      → VCC를 순간적으로 떨어뜨림 → 회로 오동작              │
│                                                             │
│   3) 광학/EM 펄스, 레이저 (FIB), 온도 등                    │
│      → 본 ChipWhisperer 플랫폼 범위 외                      │
└─────────────────────────────────────────────────────────────┘
```

### 왜 ChipWhisperer를 쓰는가?

오류주입은 **수십 ns 단위의 정밀 타이밍 제어**가 필수인데,
이를 일반 신호 발생기로 구현하기는 매우 어렵습니다.
ChipWhisperer는 다음을 한 보드에 통합한 오픈소스 플랫폼입니다:

- 타겟 MCU 클럭에 **위상 동기된** 글리치 펄스 (jitter 최소화)
- 글리치 ON/OFF · 위치 · 폭 · 외부 오프셋을 Python API로 제어
- 트리거 신호 발생부터 결과 회수까지 **하나의 연결**로 처리

### 실험 환경

```
호스트 PC (Python / Jupyter)
    │
    │  USB
    ▼
ChipWhisperer-Husky (scope + 글리치 발생기)
    │
    │  20-pin 커넥터 (CLK + GLITCH + GPIO 트리거)
    ▼
CW308 UFO 보드 — STM32F303 마이크로컨트롤러 (타겟)
```

- **scope.glitch** : 글리치 파형을 합성/주입하는 객체
- **target**       : 타겟 MCU와 SimpleSerial로 통신하는 객체
- 타겟 펌웨어 내부에는 단순한 `for`-루프 기반 OTP 연산이 있고,
  글리치로 이 루프를 **건너뛰거나(skip)** 결과 바이트를 **변조(fault)** 하는 것이 본 실습의 목표입니다.

### 📖 핵심 용어집

| 용어 | 의미 |
|:----|:----|
| **Glitch**       | 클럭/전압 파형에 의도적으로 삽입하는 짧은 외란 |
| **ext_offset**   | 트리거 신호 후 **N 클럭** 뒤에 글리치를 시작 (단위: clock cycle) |
| **offset**       | 1 클럭 내에서 **언제** 글리치 펄스를 시작할지 (위상) |
| **width**        | 1 클럭 내에서 글리치 펄스를 **얼마나 오래** 유지할지 (펄스 폭) |
| **phase_shift_steps** | offset/width 가 가질 수 있는 **최대 단계 수** (Husky 기준 ~6000) |
| **arm_timing**   | 글리치 발생 시점 — `'after_scope'`(scope arm 후) / `'no_glitch'`(글리치 비활성) |
| **clock_xor** 등 | 글리치 출력 합성 모드 (하단 표 참조) |

#### glitch.output 모드 정리

| 모드 | 동작 | 주 용도 |
|:----|:----|:----|
| `clock_only`  | 입력 클럭만 그대로 출력 | 디버그용 |
| `glitch_only` | 글리치 펄스만 출력 (클럭 X) | **전압 글리치** |
| `clock_or`    | 클럭 OR 글리치 (둘 중 하나라도 HIGH 시 HIGH) | 클럭 글리치 |
| `clock_xor`   | 클럭 XOR 글리치 (둘이 다를 때 HIGH) | **클럭 글리치 (본 실습)** |
| `enable_only` | 지정 클럭 사이클 동안만 클럭 출력 | 카운터 기반 정밀 글리치 |

#### 결과 행위 분류 (본 실습의 평가 지표)

타겟의 응답을 **6가지 카테고리**로 분류해 글리치 효과를 정량화합니다:

| 코드 | 명칭 | 의미 | 공격적 가치 |
|:----:|:----|:----|:----:|
| `0` | **freezing** | 응답 없음 / 타임아웃 | ⚠️ 너무 강함 |
| `1` | **normal**   | 정상 동작 (오류주입 효과 없음) | ❌ 무효 |
| `2` | **for-loop skip** | 출력이 짧음 → 루프 스킵 성공 | ✅ **인증 우회 용도** |
| `3` | **one faulty byte** | 1바이트만 변조됨 | ✅ **DFA 공격 용도** |
| `4` | **few faulty bytes** | 2~4바이트 변조 | ⚠️ 부분 활용 가능 |
| `5` | **etc.**     | 5바이트 이상 변조 | ⚠️ 너무 큰 변조 |

> 💡 **공격적으로 가치 있는 결과는 `2`와 `3`** 입니다.
> - `2`(loop skip)는 길이 검사 우회 등 **제어 흐름 변조** 에 쓰이고,
> - `3`(1-byte fault)는 **DFA(Differential Fault Analysis)** 키 복원에 쓰입니다.

---

## 📦 사전 준비: 플랫폼 설정 및 시각화 환경

### PLATFORM 변수 설정

사용 중인 ChipWhisperer 하드웨어 종류를 지정합니다.
본 실습은 **HUSKY + CW308_STM32F3** 환경을 기준으로 합니다.

In [1]:
# 사용 중인 ChipWhisperer 플랫폼을 지정합니다
# CWHUSKY        : ChipWhisperer-Husky (SAM4S)
# CW308_STM32F3  : CW308 UFO + STM32F303 (Pro 환경에서도 동작 가능)

# PLATFORM = 'CWHUSKY'
PLATFORM = 'CW308_STM32F3'

### 공통 설정 파일 실행

`My_Setup.ipynb` 를 실행하면 아래 작업이 자동으로 진행됩니다:

- ChipWhisperer 라이브러리 임포트 (`chipwhisperer`, `numpy`, `scipy`, `tqdm`, `matplotlib`, `logging` 등)
- `scope` / `target` 객체 생성 및 USB 연결
- 타겟 펌웨어 (FIA용 `simpleserial-base` 변형) 컴파일 및 업로드
- 보조 함수 정의 (`my_fsr_cmd`, `my_get_trace`, `my_setting_num_samples`, `reset_target` 등)

In [2]:
# 공통 설정 노트북 실행 (scope, target 객체 초기화 + 펌웨어 빌드/프로그래밍)
%run '../base/My_Setup.ipynb'

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
.
Welcome to another exciting ChipWhisperer target build!!
.
Cleaning project:
rm -f -- simpleserial-base-CW308_CC2538.hex simpleserial-base-CW301_AVR.hex simpleserial-base-CW303.hex simpleserial-base-CW304.hex simpleserial-base-CW308_MEGARF.hex simpleserial-base-CW308_SAM4L.hex simpleserial-base-CW308_STM32F0.hex simpleserial-base-CW308_STM32F1.hex simpleserial-base-CW308_STM32F2.hex simpleserial-base-CW308_STM32F3.hex simpleserial-base-CW308_STM32F4.hex simpleserial-base-CW308_K24F.hex simpleserial-base-CW308_NRF52.hex simpleserial-base-CW308_AURIX.hex simpleserial-base-CW308_SAML11.hex simpleserial-base-CW308_EFM32TG11B.hex simpleserial-base-CWLITEARM.hex simpleserial-base-CWLITEXMEGA.hex simpleserial-base-CWNANO.hex simpleserial-base-CWHUSKY.hex simpleserial-base-CW308_K82F.hex simpleserial-base-CW308_PSOC62.hex simpleserial-base-CW308_IMXRT1062.hex simpleserial-base-CW308_FE310.hex simpleserial-base-CW308_EFR32MG21A.hex simpleseria

### 🎨 시각화 환경 (Bokeh) 초기화

본 강의의 모든 파형 시각화는 **Bokeh** 을 사용합니다.
글리치 효과는 **국소적인 시점 차이**로 나타나므로 인터랙티브 확대/축소가 필수적입니다.

- 🔍 **Wheel Zoom**     : 마우스 휠로 시간축을 자유롭게 확대/축소
- 🤚 **Pan**             : 드래그로 영역 이동
- 📍 **Hover Tooltip**  : 마우스 오버 시 정확한 (sample, value) 값 표시
- 🖼 **Box Zoom / Reset / Save** : 관심 구간만 박스로 선택해 확대, 원본 복귀, PNG 저장

In [3]:
# Bokeh 시각화 초기화 (한 번만 실행하면 노트북 전체에서 인라인 출력 가능)
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import HoverTool, ColumnDataSource, Span, Label, BoxAnnotation
from bokeh.palettes import Category10, Viridis256, Spectral6
from bokeh.layouts import column, row, gridplot
from bokeh.transform import linear_cmap

output_notebook()
print('✅ Bokeh 인라인 출력이 활성화되었습니다.')

Loading BokehJS ...

✅ Bokeh 인라인 출력이 활성화되었습니다.


---

# ⚙️ 1단계 — 글리치 모듈 설정 및 ADC 클럭 동기화

> **이 단계의 목표**
> 글리치 발생기를 활성화하고, **1 ADC 샘플 = 1 타겟 클럭** 이 되도록 클럭 체인을 정렬합니다.
> 이 정렬이 정확해야 `ext_offset` (클럭 단위)과 ADC 샘플 인덱스가 1:1로 대응되어
> 파형 위에서 글리치 시점을 직관적으로 식별할 수 있습니다.

---

### 1.1 왜 1 sample = 1 clock 이어야 하는가?

오류주입 분석에서는 **"몇 번째 클럭에서 글리치가 발생하는지"** 가 가장 중요한 정보입니다.
SCA와 달리 정밀한 누설 패턴 분석이 아니라, **"이 클럭에 명령어 X가 실행 중"** 이라는
시간축 매핑이 필요하기 때문입니다.

```
ADC 샘플레이트 = 타겟 클럭 × adc_mul × decimate_inverse

  · adc_mul = 4   → 1 클럭당 4 샘플 (SCA에서 미세 누설 분석용)
  · adc_mul = 1   → 1 클럭당 1 샘플 (FIA에서 시점 식별용) ← 본 실습
  · decimate = 1  → 다운샘플링 없음
```

따라서:
- `scope.adc.trig_count` 값이 곧 **연산이 차지한 클럭 사이클 수**
- 트리거 후 N 번째 클럭 = 트리거 후 N 번째 샘플
- `ext_offset = N` 으로 설정하면 **그 시점에 정확히 글리치가 떨어짐**

### 1.2 cglitch_setup() vs vglitch_setup()

| 함수 | 활성 글리치 | 출력 핀 |
|:----|:----|:----|
| `scope.cglitch_setup()` | **클럭** 글리치 (XOR 합성) | `glitch_clk` 라인 |
| `scope.vglitch_setup()` | **전압** 글리치 (HP/LP 트랜지스터) | `crowbar` 라인 |

본 노트북은 **클럭 글리치** 기준으로 진행합니다.
전압 글리치는 공식 가이드 내용을 주석으로 첨부합니다.

In [4]:
# ── 1) 글리치 모듈 활성화 ───────────────────────────────
scope.cglitch_setup()    # clock glitch 모드
##################################################################################################
## 전압 글리치 모드 (본 실습에서는 사용 안 함) - Chipwhisperer 공식 가이드 참고자료
## 
## There are 5 ways that the glitch module can combine the clock with its glitch pulses:
## clock_only: Output only the original input clock.
## glitch_only: Output only the glitch pulses - do not use the clock.
## clock_or: Output is high if either the clock or glitch are high.
## clock_xor: Output is high if clock and glitch are different.
## enable_only: Output is high for repeat full clock cycles. width and width_fine have no effect, 
##              but offset and offset_fine do.
##
## Some of these settings are only useful in certain scenarios:
## Clock glitching: 'clock_or' or 'clock_xor
## Voltage glitching: 'glitch_only' or 'enable_only'
##
## 전압 글리치 모드 활성 코드
'''
scope.vglitch_setup('both')   
if scope._is_husky:
    scope.vglitch_setup('hp', default_setup=False) # HP alone works best for Husky
else:
    scope.vglitch_setup('both', default_setup=False) # use both transistors
'''
##################################################################################################

# ── 2) ADC 클럭 동기화: 1 샘플 = 1 클럭 ──────────────────
scope.clock.adc_src    = 'clkgen_x1'  # ADC 클럭 = 타겟 클럭 (×1)
scope.clock.clkgen_src = 'system'     # 클럭 소스: 시스템 클럭
scope.clock.adc_mul    = 1            # 오버샘플링 없음
scope.adc.decimate     = 1            # 다운샘플링 없음

# ── 3) 타겟 보드 리셋 (2회 반복으로 안정화) ──────────────
# reset_target(scope) → ../base/Setup_Generic.ipynb 정의
reset_target(scope)
time.sleep(1)
reset_target(scope)
time.sleep(1)

# ── 4) 설정 검증 ────────────────────────────────────────
print('=== ADC / 클럭 설정 ===')
print(scope.adc)
print(scope.clock)
print()
print(f'  scope.clock.adc_src    : {scope.clock.adc_src}')
print(f'  scope.clock.clkgen_src : {scope.clock.clkgen_src}')
print(f'  scope.clock.adc_mul    : {scope.clock.adc_mul}')
print(f'  scope.adc.decimate     : {scope.adc.decimate}')
print()
print('✅ 1 ADC 샘플 = 1 타겟 클럭 정렬 완료')

(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:1628) scope.clock.adc_src is provided for backwards compability, but scope.clock.clkgen_src and scope.clock.adc_mul should be used for Husky.
(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:703) Target clock may drop; you may need to reset your target.


=== ADC / 클럭 설정 ===
state                    = False
basic_mode               = rising_edge
timeout                  = 2
offset                   = 0
presamples               = 0
samples                  = 5000
decimate                 = 1
trig_count               = 0
stream_mode              = False
test_mode                = False
bits_per_sample          = 12
segments                 = 1
segment_cycles           = 0
segment_cycle_counter_en = False
clip_errors_disabled     = True
lo_gain_errors_disabled  = True
errors                   = False

clkgen_src             = system
clkgen_freq            = 7371428.571428572
adc_mul                = 1
adc_freq               = 7371428.571428572
freq_ctr               = 0
freq_ctr_src           = extclk
clkgen_locked          = True
adc_phase              = 0.0
extclk_monitor_enabled = False
extclk_error           = False
extclk_tolerance       = 1144409.1796875


  scope.clock.adc_src    : For Husky, please use scope.clock.clkgen_src and sc

> ⚠️ **CW308_STM32F3 (Pro) 사용 시 주의**
> Pro 환경에서는 `scope.glitch.offset` / `scope.glitch.width` 가 **퍼센트 단위**로 표시되며,
> 일부 셀에서 `Husky 전용` 경고가 나타날 수 있습니다.
> 스크립트 자체는 두 환경 모두에서 동작하도록 작성되어 있으므로 경고는 무시해도 됩니다.

---

# 🌊 2단계 — 베이스라인 파형 수집 및 글리치 시점 후보 식별

> **이 단계의 목표**
> 글리치를 **주입하지 않은 정상 상태**의 파형을 1개 수집해
> (a) 정상 출력값 `expected_ret` 를 확보하고,
> (b) 글리치를 떨어뜨릴 만한 **시점 후보 구간**을 파형에서 시각적으로 식별합니다.

---

### 2.1 타겟 펌웨어 동작 개요

본 실습의 펌웨어는 다음과 같은 단순한 OTP-style 연산을 수행합니다:

```c
// 의사 코드 (실제는 simpleserial-base 의 'c' 핸들러)
for (i = 0; i < global_len; i++) {
    output[i] = key[i] ^ plaintext[i];
}
```

- `global_len` = 호스트에서 `0x81 'l'` 명령으로 설정 (본 실습은 40)
- 글리치가 **for-loop의 종료 조건 검사** 에 영향을 주면 → loop skip → 출력 길이 감소
- 글리치가 **XOR 연산 자체**에 영향을 주면 → 결과 바이트가 잘못 계산됨

> 💡 **왜 단순 XOR 인가?**
> 본 강의의 목표는 **글리치 파라미터 탐색 절차** 를 익히는 것이므로,
> 펌웨어를 단순화해 글리치가 어떤 동작에 영향을 주는지 명확히 관찰할 수 있게 했습니다.
> AES 등 실제 암호 연산에 대한 DFA 공격은 이 절차를 동일하게 적용하면 됩니다.

### 2.2 테스트 데이터 준비 + 베이스라인 파형 수집

In [5]:
# ── 테스트 데이터 준비 ──────────────────────────────────
MAX_DATA_LEN = 40  # for-loop 의 반복 횟수 (= 출력 바이트 수)

random.seed(1)  # 재현성 보장
data_k = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
data_p = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))

print(f'MAX_DATA_LEN : {MAX_DATA_LEN}')
print(f'data_k (앞 16바이트) : {data_k[:16].hex(" ")} ...')
print(f'data_p (앞 16바이트) : {data_p[:16].hex(" ")} ...')

MAX_DATA_LEN : 40
data_k (앞 16바이트) : 44 20 82 3c fd e6 f1 c2 6b 30 f9 0e c7 dd 01 e4 ...
data_p (앞 16바이트) : 5f 97 3d aa d8 61 9b 91 ff c9 11 f5 7c ce d4 58 ...


In [6]:
# ── 타겟 초기화 + 데이터 주입 ───────────────────────────
reset_target(scope)
my_fsr_cmd(target, 0x81, 'k', data_k)                      # 키 주입
my_fsr_cmd(target, 0x81, 'p', data_p)                      # 평문 주입
my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))   # 길이 설정

# ── samples 자동 설정: trig_count 측정 → samples 결정 ───
# 1단계에서 1 sample = 1 clock 으로 설정했으므로
# scope.adc.samples == 연산에 걸린 총 클럭 수
my_setting_num_samples(target, scope)

# ── 베이스라인 파형 + 정상 출력값 수집 ────────────────────
expected_ret, trace = my_get_trace(target, scope)

print(f'\n=== 베이스라인 ===')
print(f'  파형 샘플 수(==클럭 수) : {scope.adc.samples}')
print(f'  expected_ret           : {expected_ret.hex(" ")}')


=== 베이스라인 ===
  파형 샘플 수(==클럭 수) : 520
  expected_ret           : 1b b7 bf 96 25 87 6a 53 94 f9 e8 fb bb 13 d5 bc 33 ca 18 42 38 58 c4 b9 39 61 28 18 ec 29 aa 21 c4 70 16 9d 5b bb 61 d8


### 2.3 글리치 파라미터 범위 산출

ChipWhisperer-Husky의 `phase_shift_steps` 가 글리치 위상 분해능의 최대값입니다.
**offset 과 width 의 허용 범위**를 미리 출력해 둡니다.

In [7]:
# ── 글리치 파라미터 범위 (Husky 기준; Pro는 퍼센트) ─────
# 클럭 1주기 내 위상:
#   · offset 0 ~ phase_shift_steps/2  : 클럭 HIGH 구간
#   · offset phase_shift_steps/2 ~ phase_shift_steps : 클럭 LOW 구간
# width  : 글리치 펄스의 폭 (최대 phase_shift_steps/2)
print(f'i_offset 범위 : 0 ~ {scope.glitch.phase_shift_steps}')
print(f'i_width  범위 : 0 ~ {scope.glitch.phase_shift_steps // 2}')
print(f'\nphase_shift_steps : {scope.glitch.phase_shift_steps}')

i_offset 범위 : 0 ~ 4592
i_width  범위 : 0 ~ 2296

phase_shift_steps : 4592


### 2.4 Bokeh으로 베이스라인 파형 시각화

수집된 정상 파형을 **인터랙티브 그래프** 로 그립니다.
파형의 패턴을 보면서 **"어느 클럭 구간에서 for-loop가 도는지"** 를 시각적으로 식별하세요.

In [8]:
# ── Bokeh 인터랙티브 파형 시각화 ────────────────────────
x = np.arange(len(trace))

source = ColumnDataSource(data=dict(x=x, y=trace))

p = figure(
    width=900, height=320,
    title='Baseline Power Trace — No Glitch (1 sample = 1 clock)',
    x_axis_label='Sample Index (= Clock Cycle from Trigger)',
    y_axis_label='Amplitude (V)',
    tools='pan,wheel_zoom,box_zoom,reset,save',
    active_scroll='wheel_zoom',
    background_fill_color='#fafafa',
    border_fill_color='white',
)

p.line('x', 'y', source=source,
       line_width=1.2, line_color='#2E86AB', line_alpha=0.9)

# 시각적 다듬기
p.title.text_font_size = '13pt'
p.title.text_color = '#2c3e50'
p.title.align = 'center'
p.grid.grid_line_alpha = 0.3
p.xaxis.axis_label_text_font_style = 'normal'
p.yaxis.axis_label_text_font_style = 'normal'
p.outline_line_color = None

# Hover 툴팁
hover = HoverTool(
    tooltips=[
        ('Clock', '@x{0}'),
        ('Amplitude', '@y{0.0000}'),
    ],
    mode='vline',
)
p.add_tools(hover)

show(p)

> 🔬 **베이스라인 파형에서 점검할 사항**
>
> - 파형의 **반복 패턴**이 보이는가? → for-loop 구간을 식별하는 단서
> - 진폭이 갑자기 변하는 구간이 있는가? → 함수 진입/종료 또는 메모리 접근
> - 트리거 직후 **수십 클럭의 셋업 구간** → `ext_offset` 의 시작값 후보
> - **루프 본체의 클럭 수** → 단일 반복(iteration)당 사이클 → loop skip 효과 추정
>
> 본 실습 펌웨어의 경우, 트리거 후 **약 170 클럭** 구간의 for-loop 연산을 타겟으로 하겠습니다.
> 따라서 `ext_offset` 의 탐색 범위를 **170 ~ +@** 로 설정합니다 (오류주입에 유효한 인스트럭션 찾기).

---

# 🎯 3단계 — 광범위 파라미터 탐색 (Parameter Sweep)

> **이 단계의 목표**
> `(ext_offset, offset, width)` 의 3차원 파라미터 공간을 격자(grid) 탐색하며,
> 각 조합마다 **여러 회 반복** 시행해 결과를 6가지 카테고리로 분류합니다.
> 이 통계가 다음 단계의 **최적 파라미터 도출** 의 기반이 됩니다.

---

### 3.1 탐색 전략

```
3차원 격자 탐색
    ext_offset : 170 ~ +@ (1 step)            ← 트리거 후 N 클럭
    offset     : 0 ~ phase_shift_steps  (100 step)  ← 1 클럭 내 시작 위상
    width      : 0 ~ phase_shift_steps/2 (100 step) ← 글리치 펄스 폭

각 (ext_offset, offset, width) 조합마다 반복 수행
    └─ 결과 분류: [freezing | normal | loop skip | 1-byte fault | few faults | etc]
```

> 💡 **반복 시행이 필요한 이유**
> 같은 파라미터라도 글리치 효과는 **확률적** 으로 발현됩니다.
> 단일 시행만으로는 우연한 성공/실패를 구분할 수 없습니다.
> 실제 연구에서는 **수 백회 이상 반복**으로 통계적 유의성을 확보합니다.

### 3.2 글리치 출력 모드 및 로깅 설정

In [9]:
# ── 글리치 발생 시점 + 출력 모드 ────────────────────────
scope.glitch.arm_timing = 'after_scope'   # scope arm 후 글리치 활성
                                          # 'no_glitch' 로 두면 글리치 비활성 (디버그용)
scope.glitch.output     = 'clock_xor'     # 클럭 XOR 글리치 (clock glitch)
                                          # · clock_or  : 클럭 OR 글리치
                                          # · glitch_only / enable_only : 전압 글리치용

# ── 경고 로그 무시 (탐색 중 빈번하게 발생하는 정상 경고) ──
# (ChipWhisperer Target  WARNING) Read timed out
# (ChipWhisperer Glitch  WARNING) Partial reconfiguration for width = 0 may not work
logging.getLogger('ChipWhisperer Target').setLevel(logging.ERROR)
logging.getLogger('ChipWhisperer Glitch').setLevel(logging.ERROR)

print('✅ 글리치 출력 모드 설정 완료')
print(f'  arm_timing : {scope.glitch.arm_timing}')
print(f'  output     : {scope.glitch.output}')

✅ 글리치 출력 모드 설정 완료
  arm_timing : after_scope
  output     : clock_xor


### 3.3 결과 저장 자료구조

탐색 결과를 **2개의 리스트**에 저장합니다:

```
cglitch_result      [코드, ext_offset, width, offset]   ← 통계 분석용 (숫자만)
cglitch_extra_data  [expected_ret, actual_ret]          ← 디버깅/검증용 (바이트)
```

- 각 행이 **1회의 시행 결과** 입니다.
- 파라미터 1조합당 여러 횟수를 반복이므로, 같은 파라미터가 여러 행에 등장합니다.
- 결과 코드 의미는 시작부분의 **결과 행위 분류표** 참조.

In [10]:
# ── 결과 저장용 리스트 ──────────────────────────────────
# cglitch_result      형식: [코드, ext_offset, width, offset]
#   코드: 0=freezing  1=normal  2=loop skip  3=1byte fault
#         4=few faults(<5)  5=etc(>=5)
# cglitch_extra_data  형식: [expected_ret, actual_ret]
cglitch_result     = []
cglitch_extra_data = []

print('자료구조 초기화 완료')

자료구조 초기화 완료


### 3.4 메인 탐색 루프

> ⏱ **실행 시간 안내**

> 아래 셀은 예제를 위한 코드(엄청 큰 step 사이즈) 입니다. 실제 공격에서는 해당 작업이 **수 시간** 이 소요될 수 있습니다.

> 실제 탐색할 때는 셀 내부의 `for i_offset in trange(...)` 와 `for i_width in trange(...)` 의

> step 값을 **작게(예: 10)** 설정해야 오류가 주입되는 최적 파라미터를 찾을 수 있습니다. (최적 파라미터 범위가 매우 좁음)

In [11]:
# ══════════════════════════════════════════════════════════
#  메인 파라미터 탐색 루프
# ══════════════════════════════════════════════════════════

for i_ext_offset in trange(170, 172, desc='i_ext_offset', leave=False):

    for i_offset in trange(0, scope.glitch.phase_shift_steps, 1000,
                           desc='i_offset', leave=False):

        for i_width in trange(0, scope.glitch.phase_shift_steps // 2, 2000,
                              desc='i_width', leave=False):

            # 유효성 검사 — 의미 없는 파라미터 건너뛰기
            if i_width == 0:
                continue
            if (i_offset + i_width) > scope.glitch.phase_shift_steps:
                continue

            # ── 동일 파라미터로 반복 시행 ───────────
            for _ in range(2):

                # 1. 글리치 파라미터 설정
                scope.glitch.ext_offset = i_ext_offset  # 트리거 후 N 클럭
                scope.glitch.offset     = i_offset      # 1 클럭 내 시작 위상
                scope.glitch.width      = i_width       # 글리치 펄스 폭

                # 2. 타겟 초기화 (매 시행마다 깨끗한 상태로 시작)
                reset_target(scope)
                my_fsr_cmd(target, 0x81, 'k', data_k)
                my_fsr_cmd(target, 0x81, 'p', data_p)
                my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))

                # 3. 글리치를 동반한 암호 연산 실행
                target.flush()
                scope.arm()
                target.send_cmd(cmd=0x82, scmd=ord('c'), data=[])
                actual_ret = target.read_cmd(timeout=500)

                # ── 결과 분류 ──────────────────────────────

                # 코드 0: 타겟 freezing (응답 없음 또는 capture 타임아웃)
                if (actual_ret is None) or (scope.capture()):
                    cglitch_result.append(
                        [0, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])
                    continue

                # 결과 회수 명령
                target.flush()
                target.send_cmd(cmd=0x83, scmd=ord('r'), data=[])
                actual_ret = target.read_cmd(timeout=500)
                if actual_ret is None:
                    cglitch_result.append(
                        [0, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])
                    continue

                # 응답 패킷 → 페이로드 추출
                actual_ret = actual_ret[3 : 3 + actual_ret[2]]

                # 코드 1: normal (오류주입이 아무 영향 없음)
                if expected_ret == actual_ret:
                    cglitch_result.append(
                        [1, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])

                # 코드 2: for-loop skip (출력 길이가 짧아짐)
                #   trig_count < (samples - 300)  →  연산이 일찍 종료됨
                elif scope.adc.trig_count < (scope.adc.samples - 300):
                    cglitch_result.append(
                        [2, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])
                    print(f'\n[코드2] for-loop skip')
                    print(f'  i_ext_offset : {scope.glitch.ext_offset}')
                    print(f'  i_width      : {scope.glitch.width}')
                    print(f'  i_offset     : {scope.glitch.offset}')
                    print(f'  expected_ret : {expected_ret.hex(" ")}')
                    print(f'  actual_ret   : {actual_ret.hex(" ")}')

                # 코드 3: 1-byte fault (DFA 공격에 가장 유용)
                elif sum(g != r for g, r in zip(expected_ret, actual_ret)) == 1:
                    cglitch_result.append(
                        [3, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])
                    print(f'\n[코드3] one faulty byte')
                    print(f'  i_ext_offset : {scope.glitch.ext_offset}')
                    print(f'  i_width      : {scope.glitch.width}')
                    print(f'  i_offset     : {scope.glitch.offset}')
                    print(f'  expected_ret : {expected_ret.hex(" ")}')
                    print(f'  actual_ret   : {actual_ret.hex(" ")}')

                # 코드 4: few faulty bytes (<5)
                elif sum(g != r for g, r in zip(expected_ret, actual_ret)) < 5:
                    cglitch_result.append(
                        [4, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])
                    print(f'\n[코드4] few faulty bytes (<5)')
                    print(f'  i_ext_offset : {scope.glitch.ext_offset}')
                    print(f'  i_width      : {scope.glitch.width}')
                    print(f'  i_offset     : {scope.glitch.offset}')
                    print(f'  expected_ret : {expected_ret.hex(" ")}')
                    print(f'  actual_ret   : {actual_ret.hex(" ")}')

                # 코드 5: 5바이트 이상 변조 (너무 큰 변조)
                else:
                    cglitch_result.append(
                        [5, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])

print('\n✅ 파라미터 탐색 완료')

i_ext_offset:   0%|          | 0/2 [00:00<?, ?it/s]

i_offset:   0%|          | 0/5 [00:00<?, ?it/s]

i_width:   0%|          | 0/2 [00:00<?, ?it/s]

i_width:   0%|          | 0/2 [00:00<?, ?it/s]

i_width:   0%|          | 0/2 [00:00<?, ?it/s]

...

i_width:   0%|          | 0/2 [00:00<?, ?it/s]

i_width:   0%|          | 0/2 [00:00<?, ?it/s]

i_width:   0%|          | 0/2 [00:00<?, ?it/s]

i_width:   0%|          | 0/2 [00:00<?, ?it/s]


✅ 파라미터 탐색 완료


> 본 실습의 펌웨어 기준으로 도출된 파라미터는 다음과 같습니다.

> 💡 **중요 포인트**
>> 같은 chipwhisperer 장비에 동일한 타겟보드라 할지라도 오류주입 최적 파라미터는 환경(케이블 길이, PC USB 전원공급 상태 등)마다 미세하게 다릅니다.

In [12]:
# ── 결과 저장용 리스트 ──────────────────────────────────
# cglitch_result      형식: [코드, ext_offset, width, offset]
#   코드: 0=freezing  1=normal  2=loop skip  3=1byte fault
#         4=few faults(<5)  5=etc(>=5)
# cglitch_extra_data  형식: [expected_ret, actual_ret]
cglitch_result     = []
cglitch_extra_data = []
loop_skip_t = []

print('자료구조 초기화 완료')

자료구조 초기화 완료


In [13]:
# ══════════════════════════════════════════════════════════
#  메인 파라미터 탐색 루프
# ══════════════════════════════════════════════════════════

for i_ext_offset in trange(170, 183, desc='i_ext_offset', leave=False):

    for i_offset in [10, 20, 30]:

        for i_width in [2090, 2100, 2110]:

            # 유효성 검사 — 의미 없는 파라미터 건너뛰기
            if i_width == 0:
                continue
            if (i_offset + i_width) > scope.glitch.phase_shift_steps:
                continue

            # ── 동일 파라미터로 반복 시행 ───────────
            for _ in range(5):

                # 1. 글리치 파라미터 설정
                scope.glitch.ext_offset = i_ext_offset  # 트리거 후 N 클럭
                scope.glitch.offset     = i_offset      # 1 클럭 내 시작 위상
                scope.glitch.width      = i_width       # 글리치 펄스 폭

                # 2. 타겟 초기화 (매 시행마다 깨끗한 상태로 시작)
                reset_target(scope)
                my_fsr_cmd(target, 0x81, 'k', data_k)
                my_fsr_cmd(target, 0x81, 'p', data_p)
                my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))

                # 3. 글리치를 동반한 암호 연산 실행
                target.flush()
                scope.arm()
                target.send_cmd(cmd=0x82, scmd=ord('c'), data=[])
                actual_ret = target.read_cmd(timeout=500)

                # ── 결과 분류 ──────────────────────────────

                # 코드 0: 타겟 freezing (응답 없음 또는 capture 타임아웃)
                if (actual_ret is None) or (scope.capture()):
                    print('.')
                    cglitch_result.append(
                        [0, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])
                    continue

                # 결과 회수 명령
                target.flush()
                target.send_cmd(cmd=0x83, scmd=ord('r'), data=[])
                actual_ret = target.read_cmd(timeout=500)
                if actual_ret is None:
                    cglitch_result.append(
                        [0, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])
                    continue

                # 응답 패킷 → 페이로드 추출
                actual_ret = actual_ret[3 : 3 + actual_ret[2]]

                # 코드 1: normal (오류주입이 아무 영향 없음)
                if expected_ret == actual_ret:
                    cglitch_result.append(
                        [1, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])

                # 코드 2: for-loop skip (출력 길이가 짧아짐)
                #   trig_count < (samples - 300)  →  연산이 일찍 종료됨
                elif scope.adc.trig_count < (scope.adc.samples - 300):
                    cglitch_result.append(
                        [2, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])
                    print(f'\n[코드2] for-loop skip')
                    print(f'  i_ext_offset : {scope.glitch.ext_offset}')
                    print(f'  i_width      : {scope.glitch.width}')
                    print(f'  i_offset     : {scope.glitch.offset}')
                    print(f'  expected_ret : {expected_ret.hex(" ")}')
                    print(f'  actual_ret   : {actual_ret.hex(" ")}')
                    loop_skip_t.append(np.array(scope.get_last_trace()))

                # 코드 3: 1-byte fault (DFA 공격에 가장 유용)
                elif sum(g != r for g, r in zip(expected_ret, actual_ret)) == 1:
                    cglitch_result.append(
                        [3, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])
                    print(f'\n[코드3] one faulty byte')
                    print(f'  i_ext_offset : {scope.glitch.ext_offset}')
                    print(f'  i_width      : {scope.glitch.width}')
                    print(f'  i_offset     : {scope.glitch.offset}')
                    print(f'  expected_ret : {expected_ret.hex(" ")}')
                    print(f'  actual_ret   : {actual_ret.hex(" ")}')

                # 코드 4: few faulty bytes (<5)
                elif sum(g != r for g, r in zip(expected_ret, actual_ret)) < 5:
                    cglitch_result.append(
                        [4, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])
                    print(f'\n[코드4] few faulty bytes (<5)')
                    print(f'  i_ext_offset : {scope.glitch.ext_offset}')
                    print(f'  i_width      : {scope.glitch.width}')
                    print(f'  i_offset     : {scope.glitch.offset}')
                    print(f'  expected_ret : {expected_ret.hex(" ")}')
                    print(f'  actual_ret   : {actual_ret.hex(" ")}')

                # 코드 5: 5바이트 이상 변조 (너무 큰 변조)
                else:
                    print('..')
                    cglitch_result.append(
                        [5, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])

# 수집한 loop skip 파형를 Bokeh으로 겹쳐 시각화
arr_t = np.array(loop_skip_t)
p = figure(
    width=900, height=320,
    title=f'{len(loop_skip_t)} Power Traces',
    x_axis_label='Sample Index',
    y_axis_label='Amplitude (V)',
    tools='pan,wheel_zoom,box_zoom,reset,save',
    active_scroll='wheel_zoom',
    background_fill_color='#fafafa',
)
palette = Category10[10]
x = np.arange(arr_t.shape[1])
for i in range(len(loop_skip_t)):
    p.line(x, arr_t[i], line_width=1.0,
            line_color=palette[i % 10], line_alpha=0.7,
            legend_label=f'Trace {i}')
p.legend.location = 'top_right'
p.legend.click_policy = 'hide'
p.title.text_font_size = '13pt'
p.grid.grid_line_alpha = 0.3
p.outline_line_color = None
show(p)

print('\n✅ 파라미터 탐색 완료')

i_ext_offset:   0%|          | 0/13 [00:00<?, ?it/s]

.
.
..

[코드3] one faulty byte
  i_ext_offset : 170
  i_width      : 2100
  i_offset     : 10
  expected_ret : 1b b7 bf 96 25 87 6a 53 94 f9 e8 fb bb 13 d5 bc 33 ca 18 42 38 58 c4 b9 39 61 28 18 ec 29 aa 21 c4 70 16 9d 5b bb 61 d8
  actual_ret   : 1b b7 bf 96 25 87 6a 53 94 f9 e8 fb dd 13 d5 bc 33 ca 18 42 38 58 c4 b9 39 61 28 18 ec 29 aa 21 c4 70 16 9d 5b bb 61 d8

[코드3] one faulty byte
  i_ext_offset : 171
  i_width      : 2090
  i_offset     : 10
  expected_ret : 1b b7 bf 96 25 87 6a 53 94 f9 e8 fb bb 13 d5 bc 33 ca 18 42 38 58 c4 b9 39 61 28 18 ec 29 aa 21 c4 70 16 9d 5b bb 61 d8
  actual_ret   : 1b b7 bf 96 25 87 6a 53 94 f9 e8 fb bb 00 d5 bc 33 ca 18 42 38 58 c4 b9 39 61 28 18 ec 29 aa 21 c4 70 16 9d 5b bb 61 d8

[코드3] one faulty byte
  i_ext_offset : 171
  i_width      : 2110
  i_offset     : 10
  expected_ret : 1b b7 bf 96 25 87 6a 53 94 f9 e8 fb bb 13 d5 bc 33 ca 18 42 38 58 c4 b9 39 61 28 18 ec 29 aa 21 c4 70 16 9d 5b bb 61 d8
  actual_ret   : 1b b7 bf 96 25 87 6a 53 94 f9 e8 


✅ 파라미터 탐색 완료


---

# 📊 4단계 — 통계 분석으로 최적 파라미터 도출

> **이 단계의 목표**
> 3단계에서 수집한 결과를 **결과 코드별로 그룹화** 하고,
> 각 그룹에서 가장 자주 등장한 `(offset, width)` 의 **최빈값(mode)** 을 찾습니다.
> 이 값이 곧 **재현성 높은 글리치 파라미터** 입니다.

---

### 4.1 numpy 배열 변환 + 그룹별 빈도 집계

In [14]:
# ── 결과 리스트 → numpy 배열 변환 ───────────────────────
cglitch_result_arr = np.array(cglitch_result, dtype=np.float64)

print(f'전체 시행 횟수 : {len(cglitch_result_arr)}')
print(f'배열 shape    : {cglitch_result_arr.shape}  (rows, [code, ext_offset, width, offset])')

전체 시행 횟수 : 585
배열 shape    : (585, 4)  (rows, [code, ext_offset, width, offset])


In [15]:
# ── 결과 코드별 빈도 + 최빈 (offset, width) ─────────────
LABELS = {
    0: 'freezing',
    1: 'normal',
    2: 'for-loop skip',
    3: 'one faulty byte',
    4: 'few faulty bytes (<5)',
    5: 'etc (>=5 faults)',
}

print('=' * 60)
print(f'{"코드":>5} {"분류":<24} {"건수":>8} {"best offset":>14} {"best width":>14}')
print('=' * 60)

for code, label in LABELS.items():
    mask = cglitch_result_arr[:, 0] == code
    cnt  = int(mask.sum())

    if cnt == 0:
        print(f'{code:>5} {label:<24} {cnt:>8} {"—":>14} {"—":>14}')
        continue

    best_offset = sp.stats.mode(cglitch_result_arr[mask, 3], keepdims=False).mode
    best_width  = sp.stats.mode(cglitch_result_arr[mask, 2], keepdims=False).mode
    print(f'{code:>5} {label:<24} {cnt:>8} {int(best_offset):>14} {int(best_width):>14}')

print('=' * 60)

   코드 분류                             건수    best offset     best width
    0 freezing                       12             10           2090
    1 normal                        553             30           2100
    2 for-loop skip                   2             10           2110
    3 one faulty byte                12             10           2090
    4 few faulty bytes (<5)           0              —              —
    5 etc (>=5 faults)                6             10           2090


> 💡 **결과 해석 가이드**
>
> - **코드 2 (for-loop skip)** 의 `best offset / width` → **인증 우회 공격**용 후보
> - **코드 3 (1-byte fault)** 의 `best offset / width` → **DFA 키 복원** 후보
> - 코드 0(freezing) 비율이 높으면 → 글리치가 너무 강함 (width를 줄일 것)
> - 코드 1(normal) 비율이 절대 다수면 → 글리치가 너무 약함 (width를 늘릴 것)

---

# 🎨 5단계 — Bokeh으로 파라미터 분포 시각화

> **이 단계의 목표**
> 모든 시행 결과를 `(offset, width)` 평면 위에 산점도로 그려, **결과 코드별 분포 영역** 을 한눈에 파악합니다.
> 같은 결과 코드가 군집을 이루는 영역이 곧 **재현성 높은 글리치 파라미터 영역** 입니다.

---

In [16]:
# ── Bokeh 산점도: (offset, width) 평면에 결과 코드별 색상 표시 ──

# 색상 팔레트 (코드 0~5)
COLOR_MAP = {
    0: '#888888',   # freezing      — gray
    1: '#1f77b4',   # normal        — blue
    2: '#2ca02c',   # loop skip     — green ✅
    3: '#d62728',   # 1-byte fault  — red ✅
    4: '#ff7f0e',   # few faults    — orange
    5: '#9467bd',   # etc           — purple
}

p = figure(
    width=900, height=500,
    title='Glitch Parameter Map — (offset, width) plane',
    x_axis_label='i_offset (1-clock 내 시작 위상)',
    y_axis_label='i_width  (글리치 펄스 폭)',
    tools='pan,wheel_zoom,box_zoom,reset,save',
    active_scroll='wheel_zoom',
    background_fill_color='#fafafa',
    border_fill_color='white',
)

# 각 결과 코드별로 별도 산점도 레이어를 그림 (legend.click_policy='hide' 활용)
for code, label in LABELS.items():
    mask = cglitch_result_arr[:, 0] == code
    if not mask.any():
        continue

    src = ColumnDataSource(data=dict(
        x=cglitch_result_arr[mask, 3],   # offset
        y=cglitch_result_arr[mask, 2],   # width
        ext=cglitch_result_arr[mask, 1], # ext_offset
        code=[code] * int(mask.sum()),
        label=[label] * int(mask.sum()),
    ))

    p.scatter('x', 'y', source=src,
              size=7, alpha=0.55,
              color=COLOR_MAP[code],
              legend_label=f'[{code}] {label} (n={int(mask.sum())})')

# 시각적 다듬기
p.title.text_font_size = '13pt'
p.title.text_color = '#2c3e50'
p.title.align = 'center'
p.grid.grid_line_alpha = 0.3
p.outline_line_color = None

# 범례: 클릭으로 토글
p.legend.location = 'top_right'
p.legend.click_policy = 'hide'
p.legend.label_text_font_size = '9pt'
p.legend.background_fill_alpha = 0.85

# Hover 툴팁
hover = HoverTool(tooltips=[
    ('result',     '@label'),
    ('i_offset',   '@x{0}'),
    ('i_width',    '@y{0}'),
    ('ext_offset', '@ext{0}'),
])
p.add_tools(hover)

show(p)

> 🔬 **시각화 활용 팁**
>
> - **범례 클릭** → 해당 결과 코드 표시/숨김 토글 (예: `normal` 만 끄면 비정상 결과만 부각됨)
> - **Box Zoom** → 관심 군집 영역만 박스로 선택해 확대
> - **Hover** → 각 점의 정확한 `(offset, width, ext_offset)` 확인
>
> 군집이 **공간적으로 연속** 이면 그 영역의 파라미터는 신뢰성이 높습니다.
> 반면 산발적으로 흩어져 있으면 우연한 성공일 가능성이 높으므로 더 많은 반복 시행이 필요합니다.

---

# 🔚 마무리 — 장비 연결 해제

노트북을 마치면 **반드시 scope / target 연결을 해제** 하세요.
그렇지 않으면 다음 세션에서 USB 연결 충돌이 발생합니다.

In [17]:
# 측정 완료 후 스코프 및 타겟 연결 해제
scope.dis()
target.dis()
print('✅ 장비 연결 해제 완료')

✅ 장비 연결 해제 완료


---

## 📝 2강 요약

| 단계 | 핵심 함수 / 명령 | 결과 |
|:----:|:---|:---|
| 1 | `scope.cglitch_setup()` + `clock.adc_mul = 1` | 1 sample = 1 clock 정렬 |
| 2 | `my_get_trace()` + Bokeh `figure().line()` | 베이스라인 파형 + `expected_ret` |
| 3 | 3중 `trange` + `scope.arm()` + `read_cmd()` | 결과 6단계 분류 |
| 4 | `np.array` + `sp.stats.mode` | 결과 코드별 최빈 `(offset, width)` |
| 5 | Bokeh `scatter()` + `legend.click_policy='hide'` | `(offset, width)` 평면 분포 시각화 |

### ✅ 2강에서 익혀야 할 핵심 개념

1. **글리치 파라미터의 의미** — `ext_offset` (클럭 단위) vs `offset / width` (1-클럭 내 위상)
2. **1 sample = 1 clock 의 중요성** — `adc_mul = 1`, `decimate = 1`
3. **결과 6단계 분류** — freezing / normal / loop skip / 1-byte / few / etc
4. **격자 탐색 + 반복 시행** 으로 통계적 유의성 확보
5. **`(offset, width)` 평면 시각화** 로 군집 영역 파악

---
*ChipWhisperer 입문 강의 — 2강 (FA 입문) 끝*